# Programowanie w języku Python

## Wykład 7 — Moduły, pakiety i środowisko uruchomieniowe Pythona

### dr inż. Waldemar Bauer

## Cel wykładu

Po dzisiejszych zajęciach powinieneś:

- rozumieć, czym jest moduł i jak działa `import`,
- wiedzieć, że import wykonuje kod modułu,
- rozumieć rolę `__name__` i `__main__`,
- wiedzieć, gdzie Python szuka modułów,
- znać znaczenie `PATH`, `PYTHONPATH`, `PYTHONSTARTUP` i `PYTHONHOME`,
- odróżniać **zmienną środowiskową** od **środowiska wirtualnego**,
- umieć utworzyć i aktywować środowisko `venv`,
- świadomie korzystać z `python -m pip`,
- rozumieć podstawową strukturę pakietu,
- znać rolę `requirements.txt` i `pyproject.toml`.

## Od funkcji do modułów

Po poprzednich wykładach potrafimy pisać funkcje:

```python
def parse_measurement(line):
    ...

def is_alarm(value, limit=30):
    ...

def format_measurement(data):
    ...
```

Gdy program rośnie, jeden plik zaczyna być trudny do utrzymania.

Naturalny kolejny krok:

> podzielić kod na kilka plików.

## Moduł

Najprościej:

> **moduł to plik `.py` zawierający kod Pythona.**

Przykład:

```text
project/
├── main.py
└── alarms.py
```

Plik `alarms.py` może zawierać funkcje, stałe i inne obiekty wykorzystywane przez `main.py`.

## Przykład modułu `alarms.py`

```python
LIMIT = 30

def is_alarm(value, limit=LIMIT):
    return value > limit
```

W innym pliku możemy użyć:

```python
import alarms

print(alarms.is_alarm(35))
```

## Co tworzy `import`?

Po:

```python
import alarms
```

nazwa `alarms` wskazuje na obiekt modułu.

Następnie:

```python
alarms.is_alarm
alarms.LIMIT
```

odwołuje się do obiektów w przestrzeni nazw modułu.

To kolejny przypadek znanego modelu:

```text
nazwa → obiekt
```

## Różne formy importu

Cały moduł:

```python
import alarms
```

Alias:

```python
import alarms as al
```

Wybrany element:

```python
from alarms import is_alarm
```

Każda forma wpływa na to, jakie nazwy pojawią się w bieżącej przestrzeni nazw.

## Dlaczego zwykle unikamy `import *`?

```python
from alarms import *
```

utrudnia odpowiedź na pytania:

- skąd pochodzi dana nazwa?
- czy dwie biblioteki nie definiują tej samej nazwy?
- jakie elementy faktycznie zostały zaimportowane?

Czytelniejszy kod zwykle jawnie pokazuje źródło importowanych obiektów.

## Import wykonuje kod modułu

Jeżeli `module.py` zawiera:

```python
print("Ładowanie modułu")

VALUE = 10
```

to:

```python
import module
```

wykona instrukcję `print()`.

Import nie jest tylko „podłączeniem pliku”.

## `__name__` — nazwa modułu

Każdy moduł ma specjalną nazwę:

```python
__name__
```

Jeżeli plik został zaimportowany:

```text
__name__ == nazwa_modułu
```

Jeżeli plik został uruchomiony bezpośrednio:

```text
__name__ == "__main__"
```

In [ ]:
print(__name__)

## Typowy punkt wejścia programu

Często spotkasz:

```python
def main():
    print("Uruchamiam program")


if __name__ == "__main__":
    main()
```

Dzięki temu kod w `main()` wykona się przy:

```bash
python program.py
```

ale nie podczas:

```python
import program
```

## Skąd Python wie, gdzie znaleźć moduł?

Polecenie:

```python
import alarms
```

uruchamia mechanizm wyszukiwania modułu.

Jednym z najważniejszych miejsc, które możemy podejrzeć, jest:

```python
sys.path
```

In [ ]:
import sys

for path in sys.path:
    print(path)

## `sys.path`

`sys.path` to lista lokalizacji, w których interpreter szuka importowanych modułów.

Jej zawartość zależy m.in. od:

- sposobu uruchomienia Pythona,
- instalacji Pythona,
- środowiska wirtualnego,
- zmiennej `PYTHONPATH`,
- konfiguracji narzędzi uruchomieniowych.

Python nie przeszukuje automatycznie całego dysku.

## Którego Pythona właśnie używam?

To jedno z najważniejszych pytań diagnostycznych:

```python
import sys
print(sys.executable)
```

Ścieżka pokaże interpreter, który rzeczywiście wykonuje program.

In [ ]:
import sys

print(sys.executable)
print(sys.version)
print(sys.prefix)
print(sys.base_prefix)

## Zmienna środowiskowa

Zmienna środowiskowa to para:

```text
NAZWA=wartość
```

przekazywana programowi przez system operacyjny.

Przykłady:

```text
PATH=...
APP_MODE=development
API_URL=...
```

Program może je odczytać w czasie działania.

In [ ]:
import os

print(os.getenv("PATH"))
print(os.getenv("APP_MODE"))
print(os.getenv("APP_MODE", "development"))

## `PATH`

`PATH` nie jest zmienną specyficzną dla Pythona.

System operacyjny używa jej do wyszukiwania programów wykonywalnych.

Gdy wpisujemy:

```bash
python
```

system musi znaleźć odpowiedni program `python` właśnie poprzez mechanizm wyszukiwania programów.

## `PATH` a `sys.executable`

Dwa różne poziomy:

```text
PATH
↓
pomaga systemowi znaleźć program python

sys.executable
↓
pokazuje, który interpreter faktycznie działa
```

To bardzo przydatne, gdy na komputerze jest kilka instalacji Pythona.

## `PYTHONPATH`

`PYTHONPATH` może dodać dodatkowe katalogi do ścieżki wyszukiwania modułów.

W uproszczeniu:

```text
PYTHONPATH
↓
wpływa na sys.path
↓
wpływa na import
```

Warto wiedzieć, jak działa, ale **nie powinien być podstawowym sposobem organizacji zwykłego projektu**.

In [ ]:
import os

print(os.getenv("PYTHONPATH"))

## `PYTHONSTARTUP`

`PYTHONSTARTUP` może wskazywać plik wykonywany przy uruchamianiu **interaktywnego interpretera** Pythona.

Przykładowe zastosowanie:

- własne importy,
- funkcje pomocnicze,
- konfiguracja sesji interaktywnej.

Nie jest to mechanizm uruchamiany przy każdym zwykłym skrypcie.

In [ ]:
print(os.getenv("PYTHONSTARTUP"))

## `PYTHONHOME`

`PYTHONHOME` pozwala wpływać na lokalizację bazowej instalacji Pythona.

Jest to zmienna bardziej zaawansowana.

### Ważne

W normalnej pracy początkującego programisty **zwykle nie ma potrzeby jej ustawiać**.

Nieprawidłowe ustawienie może spowodować problemy z uruchomieniem interpretera.

In [ ]:
print(os.getenv("PYTHONHOME"))

## Najważniejsze zmienne w jednym miejscu

```text
PATH
└── gdzie system szuka programów wykonywalnych

PYTHONPATH
└── dodatkowe miejsca wyszukiwania modułów

PYTHONSTARTUP
└── plik startowy dla interaktywnego Pythona

PYTHONHOME
└── lokalizacja bazowej instalacji Pythona
```

Za chwilę zobaczymy jeszcze:

```text
VIRTUAL_ENV
└── aktywne środowisko wirtualne
```

## Zmienna środowiskowa ≠ środowisko wirtualne

```text
ZMIENNA ŚRODOWISKOWA
PATH=...
APP_MODE=production

→ pojedyncza wartość konfiguracyjna
```

```text
ŚRODOWISKO WIRTUALNE
.venv/
├── interpreter
├── pip
└── site-packages/

→ odseparowane środowisko projektu
```

To dwa różne pojęcia.

## Po co środowisko wirtualne?

Wyobraźmy sobie dwa projekty:

```text
Projekt A
└── biblioteka X w wersji 2

Projekt B
└── biblioteka X w wersji 1
```

Oba projekty mogą działać poprawnie, ale wymagają różnych zależności.

Środowisko wirtualne pozwala je rozdzielić.

## `venv`

Python zawiera moduł `venv` do tworzenia środowisk wirtualnych.

Najczęściej:

```bash
python -m venv .venv
```

Interpretacja:

```text
python      → uruchom interpreter
-m venv     → uruchom moduł venv
.venv       → katalog nowego środowiska
```

## Dlaczego nazwa `.venv`?

Nie jest obowiązkowa, ale jest popularna, ponieważ:

- środowisko znajduje się przy projekcie,
- łatwo je rozpoznać,
- wiele IDE potrafi je automatycznie wykryć,
- katalog zaczynający się od kropki jest zwykle traktowany jako techniczny.

Środowiska wirtualnego zazwyczaj **nie dodajemy do repozytorium Git**.

## Aktywacja — Linux i macOS

Po utworzeniu:

```bash
python -m venv .venv
```

aktywacja:

```bash
source .venv/bin/activate
```

dezaktywacja:

```bash
deactivate
```

## Aktywacja — Windows PowerShell

```powershell
.venv\Scripts\Activate.ps1
```

Windows CMD:

```cmd
.venv\Scripts\activate.bat
```

Dezaktywacja:

```text
deactivate
```

## Co właściwie robi aktywacja?

Aktywacja nie „włącza nowego Pythona” magicznym przełącznikiem.

Zmienia środowisko bieżącej powłoki, przede wszystkim tak, aby katalog środowiska znalazł się wcześniej w `PATH`.

W efekcie:

```bash
python
```

wskazuje na interpreter z `.venv`.

## Sprawdzenie aktywnego środowiska

Po aktywacji możemy sprawdzić:

In [ ]:
import os
import sys

print("Python:", sys.executable)
print("Prefix:", sys.prefix)
print("Base prefix:", sys.base_prefix)
print("VIRTUAL_ENV:", os.getenv("VIRTUAL_ENV"))

## Jak rozpoznać `venv`?

Często:

```python
sys.prefix != sys.base_prefix
```

oznacza, że interpreter działa w środowisku wirtualnym.

Dodatkowo aktywacja środowiska zwykle ustawia zmienną:

```text
VIRTUAL_ENV
```

Nie należy jednak opierać całej logiki programu wyłącznie na tej zmiennej.

## Instalowanie bibliotek — `pip`

Do zarządzania pakietami instalowanymi zewnętrznie najczęściej używamy `pip`.

W tym kursie preferujemy zapis:

```bash
python -m pip install requests
```

zamiast samego:

```bash
pip install requests
```

## Dlaczego `python -m pip`?

Bo jawnie mówimy:

> uruchom moduł `pip` przy pomocy **tego konkretnego interpretera**.

To ogranicza bardzo częsty problem:

```text
python → środowisko A
pip    → środowisko B
```

## Podstawowe polecenia `pip`

```bash
python -m pip install requests
python -m pip uninstall requests
python -m pip list
python -m pip show requests
```

Instalacja określonej wersji ma składnię:

```bash
python -m pip install "package==VERSION"
```

## Biblioteka standardowa a pakiet zewnętrzny

Nie każda biblioteka wymaga instalacji.

Przykłady biblioteki standardowej:

```python
import math
import pathlib
import datetime
import json
import os
import sys
```

Są dostarczane razem z Pythonem.

Pakiet zewnętrzny trzeba zwykle wcześniej zainstalować.

## Jak poznawać moduł?

Przydatne narzędzia:

```python
dir(module)
help(module)
```

oraz przede wszystkim dokumentacja biblioteki.

In [ ]:
import math

print(math.pi)
print(dir(math)[:10])

# help(math)

## `requirements.txt`

Jednym ze sposobów zapisania zainstalowanych zależności jest:

```bash
python -m pip freeze > requirements.txt
```

Odtworzenie:

```bash
python -m pip install -r requirements.txt
```

To prosty i nadal bardzo często spotykany mechanizm.

## Uwaga na `pip freeze`

`pip freeze` zapisuje pakiety z aktualnego środowiska.

Dlatego dobry przebieg pracy to:

```text
utwórz czyste .venv
↓
zainstaluj potrzebne zależności
↓
dopiero potem zapisz requirements.txt
```

Inaczej możemy przypadkiem zapisać biblioteki niezwiązane z projektem.

## `pyproject.toml`

Współczesne projekty Pythona bardzo często posiadają plik:

```text
pyproject.toml
```

Przykład:

```toml
[project]
name = "sensor-app"
version = "0.1.0"
dependencies = [
    "requests",
]
```

Na tym etapie nie będziemy jeszcze budować własnych paczek.

## Teraz: pakiet

Gdy projekt posiada wiele modułów, możemy pogrupować je w katalog:

```text
sensor_app/
├── main.py
└── sensors/
    ├── __init__.py
    ├── parser.py
    └── alarms.py
```

Katalog `sensors` jest pakietem zawierającym moduły.

## `__init__.py`

Klasyczny pakiet posiada plik:

```text
__init__.py
```

Uwaga na nazwę:

```text
__init__.py     ✓
__init.py__     ✗
```

Python wspiera również namespace packages bez tego pliku, ale na początku kursu będziemy używać klasycznej struktury z `__init__.py`.

## Import z pakietu

Dla struktury:

```text
sensors/
├── __init__.py
├── parser.py
└── alarms.py
```

możemy użyć:

```python
from sensors.parser import parse_measurement
from sensors.alarms import is_alarm
```

## Pełna struktura projektu

Teraz wszystkie elementy zaczynają do siebie pasować:

```text
sensor_app/
│
├── .venv/              ← środowisko, NIE commitujemy
├── main.py             ← punkt wejścia
├── requirements.txt    ← zależności
├── pyproject.toml      ← konfiguracja projektu
│
└── sensors/            ← nasz pakiet
    ├── __init__.py
    ├── parser.py
    └── alarms.py
```

## Dwa katalogi, dwa różne cele

```text
.venv/
↓
środowisko wykonawcze i zależności

sensors/
↓
organizacja naszego kodu
```

To bardzo ważne rozróżnienie.

## Praktyczna diagnostyka środowiska

Gdy pojawia się:

```text
ModuleNotFoundError
```

nie zaczynaj od wielokrotnego instalowania pakietu.

Najpierw sprawdź:

In [ ]:
import os
import sys

print("Python:", sys.executable)
print("Version:", sys.version)
print("Prefix:", sys.prefix)
print("VIRTUAL_ENV:", os.getenv("VIRTUAL_ENV"))

print("\nsys.path:")
print(*sys.path, sep="\n")

## Typowa procedura przy problemie z importem

```text
1. Który interpreter działa?
   sys.executable

2. Czy działa oczekiwane .venv?
   sys.prefix / VIRTUAL_ENV

3. Gdzie Python szuka modułów?
   sys.path

4. Czy pakiet jest zainstalowany w tym interpreterze?
   python -m pip show ...

5. Czy struktura naszego projektu jest poprawna?
```

To jest bardziej skuteczne niż przypadkowe wykonywanie kolejnych `pip install`.

## Mini-projekt: przygotowanie środowiska

```bash
mkdir sensor_app
cd sensor_app

python -m venv .venv
```

Następnie aktywujemy środowisko i sprawdzamy:

```bash
python -c "import sys; print(sys.executable)"
```

## Mini-projekt: struktura kodu

```text
sensor_app/
├── main.py
└── sensors/
    ├── __init__.py
    ├── parser.py
    └── alarms.py
```

`parser.py`:

```python
def parse_measurement(line):
    sensor, kind, value, unit = line.strip().split(";")
    return sensor, kind, float(value), unit
```

## Mini-projekt: moduł alarmów

`alarms.py`:

```python
def is_alarm(value, limit=30):
    return value > limit
```

`main.py`:

```python
from sensors.parser import parse_measurement
from sensors.alarms import is_alarm
```

## Mini-projekt: punkt wejścia

```python
from sensors.parser import parse_measurement
from sensors.alarms import is_alarm


def main():
    line = "sensor_01;temperature;35.2;C"
    sensor, kind, value, unit = parse_measurement(line)

    print(sensor, kind, value, unit)

    if is_alarm(value):
        print("ALARM")


if __name__ == "__main__":
    main()
```

## Program można rozumieć na kolejnych poziomach

```text
KOD
↓
funkcje

ORGANIZACJA KODU
↓
moduły i pakiety

WYSZUKIWANIE KODU
↓
import, sys.path, PYTHONPATH

INTERPRETER
↓
sys.executable, PATH

ŚRODOWISKO PROJEKTU
↓
venv, pip, zależności
```

To kolejne poziomy tej samej aplikacji.

## Ćwiczenie

Utwórz mały projekt:

```text
weather_app/
├── main.py
└── weather/
    ├── __init__.py
    ├── parser.py
    └── rules.py
```

Następnie:

1. utwórz `.venv`,
2. aktywuj je,
3. sprawdź `sys.executable`,
4. zaimplementuj po jednej funkcji w `parser.py` i `rules.py`,
5. zaimportuj je w `main.py`,
6. dodaj `if __name__ == "__main__":`,
7. wyświetl `sys.path` i wskaż katalog odpowiadający projektowi.

## Podsumowanie

Po tym wykładzie powinieneś pamiętać:

- moduł to plik `.py`,
- `import` tworzy dostęp do przestrzeni nazw modułu i wykonuje jego kod,
- `__name__ == "__main__"` rozróżnia uruchomienie od importu,
- `sys.path` pokazuje ścieżki wyszukiwania modułów,
- `PATH` pomaga systemowi znaleźć interpreter,
- `PYTHONPATH` może rozszerzać ścieżkę importu,
- `PYTHONSTARTUP` dotyczy sesji interaktywnej,
- `PYTHONHOME` jest zaawansowaną konfiguracją instalacji,
- `venv` izoluje środowisko projektu,
- aktywacja środowiska wpływa m.in. na `PATH`,
- `python -m pip` wiąże instalację pakietów z konkretnym interpreterem,
- pakiet organizuje moduły w większą całość.

## Następny wykład

# Pliki i trwałe przechowywanie danych

Wykorzystamy:

- napisy z W6,
- moduły i środowisko z W7,
- funkcje z W4–W5.

Poznamy pracę z plikami, `pathlib`, kodowanie tekstu oraz podstawowe formaty zapisu danych.